# Final Paper Results

This notebook reads the filtered artifacts in `paper_results/`. It intentionally keeps only the selected tables and figures for the revision.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd()
if project_root.name == "comparisons":
    project_root = project_root.parent

paper_results = project_root / "paper_results"
tables_dir = paper_results / "tables"
figures_dir = paper_results / "figures"

assert tables_dir.exists(), f"Missing tables folder: {tables_dir}"
assert figures_dir.exists(), f"Missing figures folder: {figures_dir}"


## Reproducing the Artifacts

The raw datasets and long-running outputs are not versioned to keep the repository lightweight. With the data available locally, reproduce the artifacts in two stages.

Run the experiments from scratch:

```bash
DATASETS="qsar_fish_toxicity concrete airfoil winered communities star abalone winewhite cycle electric meps19 superconductivity homes protein WEC"
ABLATION_DATASETS="airfoil concrete winered winewhite meps19"

poetry run bash run_experiments.sh --full --n-rep 50 --base-model qnn \
  --gamma 0.2 --gamma-min 0.05 --gamma-max 0.9 --tau-gamma 1.0 $DATASETS

poetry run bash run_experiments.sh --outlier-only --n-rep 50 --base-model qnn --outlier-detector lof $DATASETS
poetry run bash run_experiments.sh --outlier-only --n-rep 50 --base-model qnn --outlier-detector isolation_forest $DATASETS

poetry run bash run_experiments_disentangle.sh --n-rep 50 --outlier-detector lof $DATASETS
poetry run bash run_experiments_disentangle.sh --n-rep 50 --outlier-detector isolation_forest $DATASETS

N_REP=15 MODE=both poetry run bash run_gamma_ablation.sh $ABLATION_DATASETS
```

Render `paper_results/` from existing `results/` summaries:

```bash
poetry run python comparisons/render_qnn_result_tables.py --model qnn --n-rep 50
poetry run python comparisons/render_qnn_result_tables.py --model qnn --n-rep 50 --outlier-detector isolation_forest --outlier-only
poetry run python comparisons/get_decomp_results.py --all --n_rep 50 --outlier_detector lof --plots boxplots --no_show
poetry run python comparisons/get_decomp_results.py --all --n_rep 50 --outlier_detector isolation_forest --plots boxplots --no_show
poetry run python comparisons/plot_outlier_coverage_ratio_heatmap.py --model qnn --n-rep 50 --outlier-detector lof
poetry run python comparisons/plot_outlier_coverage_ratio_heatmap.py --model qnn --n-rep 50 --outlier-detector isolation_forest
poetry run python comparisons/plot_outlier_coverage_ratio_selection.py --model qnn --n-rep 50 --outlier-detector lof --summary-only
poetry run python comparisons/plot_outlier_coverage_ratio_selection.py --model qnn --n-rep 50 --outlier-detector isolation_forest --summary-only
poetry run python comparisons/plot_scarcity_q4_coverage_smis_selection.py --model qnn --n-rep 50 --scarcity-method knn --extra-formats
poetry run python comparisons/plot_scarcity_q4_coverage_smis_selection.py --model qnn --n-rep 50 --scarcity-method isolation_forest --extra-formats
poetry run python comparisons/plot_gamma_ablation.py --study both
```


## Tables Used in the Revision


In [ ]:
DATASET_ORDER = [
    "qsar fish toxicity",
    "concrete",
    "airfoil",
    "winered",
    "communities",
    "star",
    "abalone",
    "winewhite",
    "cycle",
    "electric",
    "meps19",
    "superconductivity",
    "homes",
    "protein",
    "WEC",
]

TABLE_FILES = {
    "SMIS": "result_qnn_smis.csv",
    "Interval length": "result_qnn_interval_length.csv",
    "Marginal coverage": "result_qnn_coverage.csv",
    "Outlier coverage (LOF)": "result_qnn_coverage_outliers.csv",
    "Outlier/inlier interval-length ratio (LOF)": "result_qnn_interval_ratio_outliers_inliers.csv",
    "Outlier coverage (Isolation Forest)": "result_qnn_coverage_outliers_isolation_forest.csv",
    "Outlier/inlier interval-length ratio (Isolation Forest)": "result_qnn_interval_ratio_outliers_inliers_isolation_forest.csv",
    "Fourth scarcity-score quartile coverage": "result_qnn_scarcity_q4_coverage.csv",
    "Fourth scarcity-score quartile coverage (Isolation Forest)": "result_qnn_scarcity_q4_coverage_isolation_forest.csv",
}

def dataset_key(label):
    return str(label).split(" (")[0]

def order_table(table):
    table = table.copy()
    order = {name: idx for idx, name in enumerate(DATASET_ORDER)}
    table["_order"] = table["Dataset"].map(lambda value: order.get(dataset_key(value), len(order)))
    return table.sort_values("_order").drop(columns="_order").reset_index(drop=True)

def read_table(filename):
    path = tables_dir / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return order_table(pd.read_csv(path))

tables = {title: read_table(filename) for title, filename in TABLE_FILES.items()}

for title, table in tables.items():
    display(Markdown(f"### {title}"))
    display(table)


## LaTeX Sources


In [ ]:
TEX_FILES = {title: filename.replace(".csv", ".tex") for title, filename in TABLE_FILES.items()}

for title, filename in TEX_FILES.items():
    path = tables_dir / filename
    print(f"% {title}: {path.relative_to(project_root)}")
    print(path.read_text())
    print("\n")


## Figures Used in the Revision


In [ ]:
FIGURE_FILES = [
    "disentanglement_boxplots.png",
    "disentanglement_boxplots_isolation_forest.png",
    "outlier_coverage_ratio_heatmap_lof.png",
    "outlier_coverage_ratio_selection_summary_lof.png",
    "outlier_coverage_ratio_heatmap_isolation_forest.png",
    "outlier_coverage_ratio_selection_summary_isolation_forest.png",
    "scarcity_q4_coverage_smis_heatmap.png",
    "scarcity_q4_coverage_smis_summary.png",
    "scarcity_q4_coverage_smis_heatmap_isolation_forest.png",
    "scarcity_q4_coverage_smis_summary_isolation_forest.png",
    "gamma_fixed_ablation_curves_main.png",
    "gamma_adaptive_ablation_curves_main.png",
]

for filename in FIGURE_FILES:
    path = figures_dir / filename
    assert path.exists(), f"Missing figure: {path}"
    print(path.relative_to(project_root))


## Quick Consistency Checks


In [ ]:
all_table_files = sorted(path.name for path in tables_dir.iterdir() if path.is_file())
all_figure_files = sorted(path.name for path in figures_dir.iterdir() if path.is_file())

forbidden_names = ["formatted", "drop" + "02", "qnn" + "_mc", "cat" + "boost", "w" + "sc"]
for forbidden in forbidden_names:
    matches = [name for name in all_table_files + all_figure_files if forbidden.lower() in name.lower()]
    assert not matches, f"Unexpected files containing {forbidden}: {matches}"

assert all(name.endswith((".csv", ".tex")) for name in all_table_files), all_table_files
assert all(name.endswith(".png") for name in all_figure_files), all_figure_files

print(f"Tables: {len(all_table_files)} files")
print(f"Figures: {len(all_figure_files)} files")
